In [6]:
# ============================================
# Step 1: Import Required Libraries
# ============================================
import numpy as np
from collections import defaultdict

# ============================================
# Step 2: Read the Corpus from corpus.txt
# ============================================
with open("corpus.txt", "r") as file:
    corpus = file.read().splitlines()

# ============================================
# Step 3: Preprocess the Text
# Convert all words into lowercase and remove empty lines
# ============================================
corpus = [word.lower().strip() for word in corpus if word.strip()]

# ============================================
# Step 4: Split Words into Characters
# Add End-of-Word symbol </w>
# ============================================
vocab = {}

for word in corpus:
    chars = " ".join(list(word)) + " </w>"
    vocab[chars] = vocab.get(chars, 0) + 1

print("========== Initial Vocabulary ==========")
for k, v in vocab.items():
    print(k, ":", v)

# ============================================
# Step 5: Count Word Frequencies
# ============================================
freq = np.array(list(vocab.values()))

print("\nWord Frequencies :", freq)
print("Total Words :", np.sum(freq))
print("Maximum Frequency :", np.max(freq))

# ============================================
# Step 6 & Step 7:
# Find Adjacent Symbol Pairs and Count Pair Frequencies
# ============================================
def get_pair_counts(vocab):

    pairs = defaultdict(int)

    for word, freq in vocab.items():

        symbols = word.split()

        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pairs[pair] += freq

    return pairs

# ============================================
# Step 8, Step 9 & Step 10:
# Select Most Frequent Pair
# Merge the Pair
# Update Vocabulary
# ============================================
def merge_pair(pair, vocab):

    new_vocab = {}

    bigram = " ".join(pair)
    replacement = "".join(pair)

    for word in vocab:

        new_word = word.replace(bigram, replacement)

        new_vocab[new_word] = vocab[word]

    return new_vocab

# ============================================
# Step 11: Repeat Merge Process
# Step 12: Store Merge Rules
# ============================================
merge_rules = []

num_merges = 10

for i in range(num_merges):

    pairs = get_pair_counts(vocab)

    if not pairs:
        break

    best_pair = max(pairs, key=pairs.get)

    merge_rules.append(best_pair)

    print("\nMerge", i + 1)
    print("Best Pair :", best_pair)
    print("Frequency :", pairs[best_pair])

    vocab = merge_pair(best_pair, vocab)

# ============================================
# Step 13: Build Final Vocabulary
# ============================================
final_vocab = set()

for word in vocab:
    final_vocab.update(word.split())

print("\n========== Final Vocabulary ==========")
print(sorted(final_vocab))

print("\nVocabulary Size :", len(final_vocab))

# ============================================
# Step 14: Implement Encoder
# ============================================
def encode(word, merge_rules):

    tokens = list(word) + ["</w>"]

    for pair in merge_rules:

        i = 0

        while i < len(tokens) - 1:

            if tokens[i] == pair[0] and tokens[i + 1] == pair[1]:

                tokens = tokens[:i] + ["".join(pair)] + tokens[i + 2:]

            else:
                i += 1

    return tokens

# ============================================
# Step 15: Implement Decoder
# ============================================
def decode(tokens):

    word = "".join(tokens)

    return word.replace("</w>", "")

# ============================================
# Step 16: Test the Tokenizer
# ============================================
test_word = "newest"

encoded = encode(test_word, merge_rules)

decoded = decode(encoded)

# ============================================
# Step 17: Display Results
# ============================================
print("\n========== Merge Rules ==========")

for i, rule in enumerate(merge_rules, 1):
    print(i, ":", rule)

print("\nTest Word :", test_word)

print("Encoded :", encoded)

print("Decoded :", decoded)

print("\n========== Program Completed ==========")

========== Initial Vocabulary ==========
l o w </w> : 1
l o w e r </w> : 1
l o w e s t </w> : 1
n e w </w> : 1
n e w e r </w> : 1
w i d e s t </w> : 1

Word Frequencies : [1 1 1 1 1 1]
Total Words : 6
Maximum Frequency : 1

Merge 1
Best Pair : ('l', 'o')
Frequency : 3

Merge 2
Best Pair : ('lo', 'w')
Frequency : 3

Merge 3
Best Pair : ('low', 'e')
Frequency : 2

Merge 4
Best Pair : ('r', '</w>')
Frequency : 2

Merge 5
Best Pair : ('s', 't')
Frequency : 2

Merge 6
Best Pair : ('st', '</w>')
Frequency : 2

Merge 7
Best Pair : ('n', 'e')
Frequency : 2

Merge 8
Best Pair : ('ne', 'w')
Frequency : 2

Merge 9
Best Pair : ('low', '</w>')
Frequency : 1

Merge 10
Best Pair : ('lowe', 'r</w>')
Frequency : 1

========== Final Vocabulary ==========
['</w>', 'd', 'e', 'i', 'low</w>', 'lowe', 'lower</w>', 'new', 'r</w>', 'st</w>', 'w']

Vocabulary Size : 11

========== Merge Rules ==========
1 : ('l', 'o')
2 : ('lo', 'w')
3 : ('low', 'e')
4 : ('r', '</w>')
5 : ('s', 't')
6 : ('st', '</w>')
7 : ('n',

In [8]:
# ============================================
# AUTOREGRESSIVE CAUSAL LANGUAGE MODEL
# ============================================

import numpy as np

print("\n========== AUTOREGRESSIVE LANGUAGE MODEL ==========")

# --------------------------------------------
# Step 1 : Build Token ID Dictionary
# --------------------------------------------

print("\nStep 1 : Token to ID Mapping")

token_to_id = {}

for i, token in enumerate(sorted(final_vocab)):
    token_to_id[token] = i

id_to_token = {v:k for k,v in token_to_id.items()}

print(token_to_id)


# --------------------------------------------
# Step 2 : Convert Encoded Tokens into IDs
# --------------------------------------------

print("\nStep 2 : Token IDs")

token_ids = []

for token in encoded:
    token_ids.append(token_to_id[token])

print("Encoded Tokens :", encoded)
print("Token IDs :", token_ids)


# --------------------------------------------
# Step 3 : Create Input and Target Pairs
# --------------------------------------------

print("\nStep 3 : Input Target Pairs")

inputs = token_ids[:-1]

targets = token_ids[1:]

print("Inputs :", inputs)
print("Targets :", targets)


# --------------------------------------------
# Step 4 : Create Embedding Matrix
# --------------------------------------------

print("\nStep 4 : Embedding Matrix")

embedding_size = 4

embedding_matrix = np.random.rand(len(token_to_id), embedding_size)

print(embedding_matrix)


# --------------------------------------------
# Step 5 : Get Input Embeddings
# --------------------------------------------

print("\nStep 5 : Input Embeddings")

input_embeddings = embedding_matrix[inputs]

print(input_embeddings)


# --------------------------------------------
# Step 6 : Positional Encoding
# --------------------------------------------

print("\nStep 6 : Positional Encoding")

positions = np.arange(len(inputs)).reshape(-1,1)

input_embeddings = input_embeddings + positions

print(input_embeddings)


# --------------------------------------------
# Step 7 : Apply Causal Mask
# --------------------------------------------

print("\nStep 7 : Causal Mask")

mask = np.tril(np.ones((len(inputs),len(inputs))))

print(mask)


# --------------------------------------------
# Step 8 : Self Attention
# --------------------------------------------

print("\nStep 8 : Self Attention")

attention_scores = np.dot(input_embeddings,input_embeddings.T)

attention_scores = attention_scores * mask

print(attention_scores)


# --------------------------------------------
# Step 9 : Linear Layer
# --------------------------------------------

print("\nStep 9 : Linear Layer")

weights = np.random.rand(embedding_size,len(token_to_id))

linear_output = np.dot(input_embeddings,weights)

print(linear_output)

# --------------------------------------------
# Step 10 : Softmax
# --------------------------------------------

print("\nStep 10 : Softmax")

exp_scores = np.exp(linear_output - np.max(linear_output, axis=1, keepdims=True))
softmax = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

print(softmax)


# --------------------------------------------
# Step 11 : Cross Entropy Loss
# --------------------------------------------

print("\nStep 11 : Cross Entropy Loss")

loss = 0

for i in range(len(targets)):
    loss += -np.log(softmax[i][targets[i]] + 1e-9)

loss = loss / len(targets)

print("Loss :", loss)


# --------------------------------------------
# Step 12 : Backpropagation (Simplified)
# --------------------------------------------

print("\nStep 12 : Backpropagation")

learning_rate = 0.01

weights = weights - learning_rate

print("Learning Rate :", learning_rate)
print("Weights Updated Successfully")


# --------------------------------------------
# Step 13 : Next Token Prediction
# --------------------------------------------

print("\nStep 13 : Next Token Prediction")

predicted_ids = np.argmax(softmax, axis=1)

print("Predicted Token IDs :", predicted_ids)

predicted_tokens = []

for pid in predicted_ids:
    predicted_tokens.append(id_to_token[pid])

print("Predicted Tokens :", predicted_tokens)


# --------------------------------------------
# Step 14 : Text Generation
# --------------------------------------------

print("\nStep 14 : Text Generation")

generated_tokens = encoded.copy()
print("Note : Generated text is based on random weights because the model is not trained.")

# Add predicted token three times
for i in range(3):
    generated_tokens.append(predicted_tokens[-1])

print("Generated Tokens :", generated_tokens)

# Join tokens into one string
generated_text = "".join(generated_tokens)

# Remove End-of-Word symbols
generated_text = generated_text.replace("</w>", "")

print("Generated Text :", generated_text)


print("\n========== AUTOREGRESSIVE LANGUAGE MODEL COMPLETED ==========")


========== AUTOREGRESSIVE LANGUAGE MODEL ==========

Step 1 : Token to ID Mapping
{'</w>': 0, 'd': 1, 'e': 2, 'i': 3, 'low</w>': 4, 'lowe': 5, 'lower</w>': 6, 'new': 7, 'r</w>': 8, 'st</w>': 9, 'w': 10}

Step 2 : Token IDs
Encoded Tokens : ['new', 'e', 'st</w>']
Token IDs : [7, 2, 9]

Step 3 : Input Target Pairs
Inputs : [7, 2]
Targets : [2, 9]

Step 4 : Embedding Matrix
[[0.51253    0.25283495 0.64140783 0.01763409]
 [0.663641   0.37848718 0.53553249 0.61062297]
 [0.50116916 0.04255602 0.91421055 0.5349252 ]
 [0.41520194 0.71994266 0.47653779 0.85598289]
 [0.03021828 0.79439811 0.04568784 0.82815037]
 [0.08275959 0.30238622 0.85331113 0.01053688]
 [0.59851346 0.09718691 0.21846004 0.4349175 ]
 [0.86960109 0.81166131 0.89735751 0.56495653]
 [0.23697232 0.16593907 0.82523702 0.72181516]
 [0.22447945 0.98183854 0.06509112 0.63805431]
 [0.12466232 0.74175619 0.26901799 0.81750358]]

Step 5 : Input Embeddings
[[0.86960109 0.81166131 0.89735751 0.56495653]
 [0.50116916 0.04255602 0.9142105